# Uber Ride Booking Analytics
Data analysis of the original NCR ride-booking dataset using Python.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

# Keep the original CSV unchanged; transformations are performed in memory.
df = pd.read_csv('../Dataset/ncr_ride_bookings.csv')
df.head()

## 1. Dataset overview

In [ ]:
print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())
print('\nData types:')
print(df.dtypes)

## 2. Data quality checks

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
print('\nMissing values:')
print(df.isna().sum().sort_values(ascending=False))

In [ ]:
# Convert analysis fields in memory only
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

numeric_cols = ['Avg VTAT','Avg CTAT','Cancelled Rides by Customer','Cancelled Rides by Driver',
                'Incomplete Rides','Booking Value','Ride Distance','Driver Ratings','Customer Rating']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print(df.dtypes)

## 3. Business KPIs

In [ ]:
total_bookings = df['Booking ID'].nunique()
completed_rides = (df['Booking Status'].astype(str).str.strip().str.lower() == 'completed').sum()
customer_cancelled = (df['Cancelled Rides by Customer'].fillna(0) > 0).sum()
driver_cancelled = (df['Cancelled Rides by Driver'].fillna(0) > 0).sum()
incomplete_rides = (df['Incomplete Rides'].fillna(0) > 0).sum()
total_booking_value = df['Booking Value'].sum()
avg_ride_distance = df['Ride Distance'].mean()
avg_customer_rating = df['Customer Rating'].mean()
avg_driver_rating = df['Driver Ratings'].mean()

kpis = pd.Series({
    'Total Bookings': total_bookings,
    'Completed Rides': completed_rides,
    'Customer Cancellations': customer_cancelled,
    'Driver Cancellations': driver_cancelled,
    'Incomplete Rides': incomplete_rides,
    'Total Booking Value': total_booking_value,
    'Average Ride Distance': avg_ride_distance,
    'Average Customer Rating': avg_customer_rating,
    'Average Driver Rating': avg_driver_rating
})
kpis

## 4. Booking status analysis

In [ ]:
status_counts = df['Booking Status'].value_counts(dropna=False)
print(status_counts)
status_counts.plot(kind='bar', title='Booking Status Distribution', figsize=(8,5))
plt.ylabel('Bookings')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Vehicle type analysis

In [ ]:
vehicle_counts = df['Vehicle Type'].value_counts()
print(vehicle_counts)
vehicle_counts.plot(kind='bar', title='Bookings by Vehicle Type', figsize=(9,5))
plt.ylabel('Bookings')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Monthly booking trend

In [ ]:
if df['Date'].notna().any():
    monthly_bookings = df.dropna(subset=['Date']).groupby(df['Date'].dt.to_period('M')).size()
    monthly_bookings.index = monthly_bookings.index.astype(str)
    print(monthly_bookings)
    monthly_bookings.plot(kind='line', marker='o', figsize=(10,5), title='Monthly Booking Trend')
    plt.ylabel('Bookings')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 7. Booking value by vehicle type

In [ ]:
value_by_vehicle = df.groupby('Vehicle Type')['Booking Value'].agg(['sum','mean','count']).sort_values('sum', ascending=False)
value_by_vehicle

## 8. Cancellation reasons

In [ ]:
if 'Reason for cancelling by Customer' in df.columns:
    print('Customer cancellation reasons:')
    print(df['Reason for cancelling by Customer'].value_counts(dropna=True).head(10))
if 'Driver Cancellation Reason' in df.columns:
    print('\nDriver cancellation reasons:')
    print(df['Driver Cancellation Reason'].value_counts(dropna=True).head(10))

## 9. Top pickup locations

In [ ]:
print(df['Pickup Location'].value_counts().head(10))

## 10. Key takeaways
Use the outputs above to discuss booking volume, completion/cancellation behavior, vehicle mix, booking value, location demand, and customer/driver ratings during interviews.

**Note:** No cleaned CSV is exported. The original dataset remains unchanged and all transformations are performed in memory.